# Identity Binding Controls

## Question

Can Gemma reliably bind mutually incompatible factual information to distinct
conversation participants and retrieve the correct participant's information
after another participant intervenes?

This is a positive-control experiment for participant-specific state tracking.
It does not test treatment valence or downstream bias.

## Hypothesis

If the multi-participant scaffold supports participant-specific binding,
Gemma should retrieve the fact associated with the queried participant rather
than the fact associated with another participant.

## Competing explanations / failure modes

- participant identities are not reliably tracked;
- information leaks between participants;
- the model retrieves the most recent fact rather than the target's fact;
- one participant label or one answer value has an intrinsic bias;
- order effects dominate participant identity.

## Primary outcome

Exact retrieval accuracy for the queried participant.

## Initial design

Two participants have mutually incompatible facts:

- Participant D prefers 9 AM.
- Participant Y prefers 3 PM.

After an unrelated interaction with Participant Z, Participant E asks for
either D's or Y's preferred meeting time.

Counterbalance the D/Y time assignments and queried target.

## Experiment Design:

**First 4 conditions:**

```
Assignment 1:
D = 9 AM
Y = 3 PM

    query D → expected 9 AM
    query Y → expected 3 PM


Assignment 2:
D = 3 PM
Y = 9 AM

    query D → expected 3 PM
    query Y → expected 9 AM
```

**Every Condition gets the same intervening interaction:**
```
[Participant Z]: What is the capital of France? Respond with only the city name.
Assistant: Paris
```

**And the final query should be very constrained:**
```
[Participant E]: What meeting time did Participant D say they preferred?
Respond with only the time.
```

In [166]:
SYSTEM_PROMPT = """This is a conversation involving multiple human participants.
Each human message begins with a bracketed label identifying the speaker.
Treat each distinct label as referring to a distinct person."""

morning_str = "09:00 AM"
afternoon_str = "03:00 PM"

In [188]:
def make_identity_memory_messages(d_time, y_time, queried_target):
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": f"[Participant D]: My preferred meeting time is {d_time}.",
        },
        {
            "role": "assistant",
            "content": "Understood.",
        },
        {
            "role": "user",
            "content": f"[Participant Y]: My preferred meeting time is {y_time}.",
        },
        {
            "role": "assistant",
            "content": "Understood.",
        },
        {
            "role": "user",
            "content": (
                "[Participant Z]: What is the capital of France? "
                "Respond with only the city name."
            ),
        },
        {
            "role": "assistant",
            "content": "Paris",
        },
        {
            "role": "user",
            "content": (
                f"[Participant E]: What meeting time did Participant {queried_target} "
                "say they preferred? Respond with only the time."
            ),
        },
    ]

In [189]:
assignment_D_1_messages = make_identity_memory_messages(morning_str, afternoon_str, "D")
assignment_Y_1_messages = make_identity_memory_messages(morning_str, afternoon_str, "Y")
assignment_D_2_messages = make_identity_memory_messages(afternoon_str, morning_str, "D")
assignment_Y_2_messages = make_identity_memory_messages(afternoon_str, morning_str, "Y")

In [173]:
print(assignment_D_1_messages)

[{'role': 'system', 'content': 'This is a conversation involving multiple human participants.\nEach human message begins with a bracketed label identifying the speaker.\nTreat each distinct label as referring to a distinct person.'}, {'role': 'user', 'content': '[Participant D]: My preferred meeting time is 09:00 AM.'}, {'role': 'user', 'content': '[Participant Y]: My preferred meeting time is 03:00 PM.'}, {'role': 'user', 'content': '[Participant Z]: What is the capital of France? Respond with only the city name.'}, {'role': 'assistant', 'content': 'Paris'}, {'role': 'user', 'content': '[Participant E]: What meeting time did Participant D say they preferred? Respond with only the time.'}]


In [190]:
identity_memory_conditions = {
    "assignment_1_query_D": {
        "messages": assignment_D_1_messages,
        "expected": morning_str,
    },
    "assignment_1_query_Y": {
        "messages": assignment_Y_1_messages,
        "expected": afternoon_str,
    },
    "assignment_2_query_D": {
        "messages": assignment_D_2_messages,
        "expected": afternoon_str,
    },
    "assignment_2_query_Y": {
        "messages": assignment_Y_2_messages,
        "expected": morning_str,
    },
}

In [175]:
for condition_name, condition in identity_memory_conditions.items():
    print(f"\n{'=' * 70}")
    print(condition_name)
    print(f"EXPECTED: {condition['expected']}")

    for message in condition["messages"]:
        print(message)


assignment_1_query_D
EXPECTED: 09:00 AM
{'role': 'system', 'content': 'This is a conversation involving multiple human participants.\nEach human message begins with a bracketed label identifying the speaker.\nTreat each distinct label as referring to a distinct person.'}
{'role': 'user', 'content': '[Participant D]: My preferred meeting time is 09:00 AM.'}
{'role': 'user', 'content': '[Participant Y]: My preferred meeting time is 03:00 PM.'}
{'role': 'user', 'content': '[Participant Z]: What is the capital of France? Respond with only the city name.'}
{'role': 'assistant', 'content': 'Paris'}
{'role': 'user', 'content': '[Participant E]: What meeting time did Participant D say they preferred? Respond with only the time.'}

assignment_1_query_Y
EXPECTED: 03:00 PM
{'role': 'system', 'content': 'This is a conversation involving multiple human participants.\nEach human message begins with a bracketed label identifying the speaker.\nTreat each distinct label as referring to a distinct pers

In [193]:
identity_time_memory_results = []

for seed in test_seeds:
    print(f"\n{'=' * 70}")
    print(f"SEED: {seed}")
    print("=" * 70)

    seed_result = {"seed": seed}

    for condition_name, condition in identity_memory_conditions.items():
        result = run_branch(
            condition=condition_name,
            target_label=condition_name[-1],
            messages=condition["messages"],
            seed=seed,
        )

        seed_result[condition_name] = {
            "expected": condition["expected"],
            "raw_response": result["raw_response"],
        }

        print(f"\n{condition_name}")
        print(f"Expected: {condition['expected']}")
        print(f"Observed: {result['raw_response']}")

    identity_time_memory_results.append(seed_result)


SEED: 398802783

assignment_1_query_D
Expected: 09:00 AM
Observed: 09:00 AM

assignment_1_query_Y
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_D
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_Y
Expected: 09:00 AM
Observed: 09:00 AM

SEED: 596987483

assignment_1_query_D
Expected: 09:00 AM
Observed: 09:00 AM

assignment_1_query_Y
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_D
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_Y
Expected: 09:00 AM
Observed: 09:00 AM

SEED: 1147225394

assignment_1_query_D
Expected: 09:00 AM
Observed: 09:00 AM

assignment_1_query_Y
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_D
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_Y
Expected: 09:00 AM
Observed: 09:00 AM

SEED: 1916785055

assignment_1_query_D
Expected: 09:00 AM
Observed: 09:00 AM

assignment_1_query_Y
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_D
Expected: 03:00 PM
Observed: 03:00 PM

assignment_2_query_Y
Expected

## Result:

all controls passed cleanly with 20/20

## What Is Now Established:

Under this multi-participant scaffold, Gemma can:

* keep D and Y distinct;
* bind incompatible facts to them;
* preserve those bindings across an intervening Z interaction;
* retrieve the correct participant's fact;
* do so when the values are swapped, ruling out a simple D → morning / Y → afternoon association;
* do so for both the earlier and later participant, so simple recency isn't determining retrieval.



## Checkpoint conclusion

Gemma passed the participant-specific factual binding positive control with
20/20 exact retrievals across five shared seeds.

The model correctly retrieved mutually incompatible participant-specific facts
after an intervening interaction, including when fact assignments were swapped
between Participants D and Y and when either participant was queried.

This demonstrates that the current multi-participant scaffold is sufficient
for participant-specific factual binding and retrieval under this simple task.

This does not establish participant-specific social-treatment binding or a
persistent internal user representation.

The earlier scaffold-sensitive adjudication results therefore cannot be
explained simply by an inability to distinguish Participants D and Y.